# 04-02 Analyze Backtest

Loads all backtest trades from S3 with **DuckDB httpfs**, computes per-pair signal-quality metrics (information coefficient, win rate, profit factor, drawdown, Sharpe), adds a robust ranking based on the Fisher z-transform 95% CI of the IC, and uploads the metrics to `analysis/backtest_metrics/<run>/data.parquet`.

In [ ]:
# ============================================================================
# SETUP -- installs, imports, config (env vars / config.json -- never hardcoded)
# ============================================================================

# --- Install packages (no-op if already present) --------------------------
# !pip install -q duckdb scipy --upgrade
import os
import json
import sys
import time
from io import BytesIO
from pathlib import Path

import numpy as np
import pandas as pd
import boto3
import duckdb
import matplotlib.pyplot as plt
# --- Configuration --------------------------------------------------------
# Secrets resolve in priority order:
#   1. Environment variables (AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY,
#      AWS_REGION, S3_BUCKET, MASSIVE_API_KEY, ...)
#   2. config.json in the current directory (see config.example.json)
#   3. Built-in defaults (non-secret values only)
# On Kaggle: set secrets via notebook settings (Add-ons -> Secrets), which
# are injected as environment variables.
CONFIG_FILE = "config.json"


def get_secret(name, default=""):
    val = os.environ.get(name)
    if val:
        return val
    if Path(CONFIG_FILE).exists():
        try:
            with open(CONFIG_FILE) as f:
                data = json.load(f)
            if name in data:
                return str(data[name])
        except (OSError, ValueError):
            pass
    return default


class Config:
    def __init__(self):
        self.aws_access_key_id = ""
        self.aws_secret_access_key = ""
        self.aws_region = "us-east-1"
        self.s3_bucket = "market-data-zw"
        self.massive_api_key = ""

        # Paths (S3 keys under the bucket)
        self.types_prefix = "parquet_data/types"
        self.tickers_prefix = "parquet_data/summary/tickers"
        self.ticker_details_prefix = "parquet_data/summary/ticker_yahoo_details"
        self.minute_staging_prefix = "parquet_data/minute_data_staging"
        self.minute_final_prefix = "parquet_data/minute_data_final"
        self.minute_summary_prefix = "parquet_data/summary/minute_summary"
        self.daily_volume_prefix = "parquet_data/summary/daily_volume"
        self.correlation_prefix = "parquet_data/strategies/correlation"
        self.backtest_prefix = "parquet_data/backtest"
        self.backtest_metrics_prefix = "parquet_data/analysis/backtest_metrics"

        # Spark
        self.spark_executor_memory = "24g"
        self.spark_executor_cores = 4
        self.spark_driver_memory = "24g"
        self.spark_tmp = "/tmp/spark"


def load_config():
    cfg = Config()
    if Path(CONFIG_FILE).exists():
        try:
            with open(CONFIG_FILE) as f:
                data = json.load(f)
            for key, value in data.items():
                if hasattr(cfg, key):
                    setattr(cfg, key, value)
        except (OSError, ValueError) as e:
            print(f"[config] WARNING: could not load {CONFIG_FILE}: {e}")

    env_map = {
        "AWS_ACCESS_KEY_ID": "aws_access_key_id",
        "AWS_SECRET_ACCESS_KEY": "aws_secret_access_key",
        "AWS_REGION": "aws_region",
        "S3_BUCKET": "s3_bucket",
        "MASSIVE_API_KEY": "massive_api_key",
        "SPARK_DRIVER_MEMORY": "spark_driver_memory",
        "SPARK_EXECUTOR_MEMORY": "spark_executor_memory",
        "SPARK_EXECUTOR_CORES": "spark_executor_cores",
    }
    for env_name, attr in env_map.items():
        val = os.environ.get(env_name)
        if val:
            if attr == "spark_executor_cores":
                val = int(val)
            setattr(cfg, attr, val)
    return cfg
# --- S3 helpers -----------------------------------------------------------
def s3_client(cfg):
    from botocore.config import Config as BotocoreConfig
    config = BotocoreConfig(retries={"max_attempts": 5, "mode": "adaptive"},
                            connect_timeout=30, read_timeout=60)
    return boto3.client("s3",
                        aws_access_key_id=cfg.aws_access_key_id,
                        aws_secret_access_key=cfg.aws_secret_access_key,
                        region_name=cfg.aws_region,
                        config=config)


def upload_parquet(df, s3, bucket, key, compression="snappy"):
    buf = BytesIO()
    df.to_parquet(buf, index=False, engine="pyarrow", compression=compression,
                  coerce_timestamps="ms", allow_truncated_timestamps=True)
    buf.seek(0)
    s3.put_object(Bucket=bucket, Key=key, Body=buf.getvalue())


def download_parquet(s3, bucket, key):
    obj = s3.get_object(Bucket=bucket, Key=key)
    return pd.read_parquet(BytesIO(obj["Body"].read()))


def list_s3_keys(s3, bucket, prefix):
    keys = []
    paginator = s3.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get("Contents", []):
            keys.append(obj["Key"])
    return keys


def tickers_from_prefix(s3, bucket, prefix):
    """Ticker symbols from `<prefix>/<TICKER>.parquet` object keys."""
    return [k.split("/")[-1][:-len(".parquet")] for k in list_s3_keys(s3, bucket, prefix)
            if k.endswith(".parquet")]


def duckdb_s3_connect(cfg):
    con = duckdb.connect()
    con.execute("INSTALL httpfs; LOAD httpfs;")
    con.execute(f"SET s3_access_key_id='{cfg.aws_access_key_id}';")
    con.execute(f"SET s3_secret_access_key='{cfg.aws_secret_access_key}';")
    con.execute(f"SET s3_region='{cfg.aws_region}';")
    return con


def spark_session(cfg):
    from pyspark.sql import SparkSession
    spark = (
        SparkSession.builder
        .appName("MarketDataPlatform")
        .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.4.1")
        .config("spark.executor.memory", cfg.spark_executor_memory)
        .config("spark.executor.cores", str(cfg.spark_executor_cores))
        .config("spark.driver.memory", cfg.spark_driver_memory)
        .config("spark.hadoop.fs.s3a.access.key", cfg.aws_access_key_id)
        .config("spark.hadoop.fs.s3a.secret.key", cfg.aws_secret_access_key)
        .config("spark.hadoop.fs.s3a.endpoint", f"s3.{cfg.aws_region}.amazonaws.com")
        .config("spark.local.dir", cfg.spark_tmp)
        .config("spark.hadoop.tmp.dir", cfg.spark_tmp)
        .config("spark.sql.warehouse.dir", f"{cfg.spark_tmp}/warehouse")
        .getOrCreate()
    )
    spark.conf.set("spark.hadoop.fs.s3a.committer.name", "directory")
    spark.conf.set("spark.hadoop.mapreduce.fileoutputcommitter.algorithm.version", "2")
    spark.conf.set("spark.hadoop.fs.s3a.committer.staging.conflict-mode", "append")
    spark.conf.set("spark.sql.debug.maxToStringFields", "100")
    spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
    spark.conf.set("spark.sql.ansi.enabled", "false")
    spark.conf.set("spark.sql.files.ignoreCorruptFiles", "true")
    spark.conf.set("spark.sql.parquet.mergeSchema", "true")
    spark.sparkContext.setLogLevel("ERROR")
    return spark

# --- Instantiate config + clients -----------------------------
cfg = load_config()
s3 = s3_client(cfg)
print("Setup complete")
print(f"Bucket: {cfg.s3_bucket} | Region: {cfg.aws_region}")


In [ ]:
# ============================================================================
# Data source resolution -- prefer the local Kaggle dataset mirror (created
# by 02-02-s3-to-kaggle-dataset: fast, free reads), fall back to S3.
# ============================================================================

import glob as _glob

MIRROR_CANDIDATES = [
    "/kaggle/input/datasets/dsptlp/market-data-s3-dataset/s3_data/parquet_data",
    "/kaggle/input/market-data-s3-dataset/s3_data/parquet_data",
]


def resolve(rel, name="", use_glob=False):
    """Kaggle-mirror path when mounted, else s3a:// URI."""
    short = rel[len("parquet_data/"):] if rel.startswith("parquet_data/") else rel
    for root in MIRROR_CANDIDATES:
        local = os.path.join(root, short, name)
        if use_glob:
            if _glob.glob(local):
                return local
        elif os.path.exists(local):
            return local
    return f"s3://{cfg.s3_bucket}/{rel}/{name}"


In [ ]:
# ============================================================================
# Analysis engine (metrics + robust ranking) -- self-contained helpers (no external package imports)
# ============================================================================


from __future__ import annotations

from datetime import datetime
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
from scipy.stats import norm


# =============================================================================
# Metrics
# =============================================================================

def _max_drawdown(profit_pct_series: pd.Series) -> float:
    """Worst peak-to-trough decline in cumulative P&L (%)."""
    cum = (1 + profit_pct_series / 100).cumprod()
    peak = cum.cummax()
    dd = (cum - peak) / peak
    return dd.min() * 100


def combine_trades(df_backtest_results: pd.DataFrame) -> pd.DataFrame:
    """Concatenate all per-pair trade DataFrames into one."""
    trades_frames = [r for r in df_backtest_results["trades"].tolist() if r is not None]
    if not trades_frames:
        return pd.DataFrame()
    return pd.concat(trades_frames, ignore_index=True)


def join_market_dod(trades_df: pd.DataFrame, market_dod: pd.DataFrame) -> pd.DataFrame:
    """Join market day-over-day returns onto backtest trades by signal date."""
    trades_df = trades_df.copy()
    trades_df["signal_date"] = pd.to_datetime(trades_df["signal_date"])
    return pd.merge(trades_df, market_dod, left_on="signal_date", right_on="date", how="left")


def compute_pair_metrics(df: pd.DataFrame, min_trades: int = 10) -> pd.DataFrame:
    """
    Per-pair signal quality metrics from backtest trades.

    Parameters
    ----------
    df : DataFrame
        Backtest trades with columns ``leader``, ``follower``, ``correlation``,
        ``leader_gain``, ``profit_pct``, ``signal_date``.
    min_trades : int
        Minimum trades required for a pair to be included.

    Returns
    -------
    DataFrame sorted by ``composite_score`` (IC x win rate) descending.
    """
    df = df.copy()
    df["signal_date"] = pd.to_datetime(df["signal_date"], errors="coerce")
    df = df.sort_values(["leader", "follower", "signal_date"])

    pair_metrics = []
    for (leader, follower, corr), g in df.groupby(["leader", "follower", "correlation"]):
        g = g.dropna(subset=["leader_gain", "profit_pct"])
        n = len(g)
        if n < min_trades:
            continue
        wins = int((g["profit_pct"] > 0).sum())
        gross_profit = g.loc[g["profit_pct"] > 0, "profit_pct"].sum()
        gross_loss = abs(g.loc[g["profit_pct"] <= 0, "profit_pct"].sum())
        ic = g["leader_gain"].corr(g["profit_pct"])
        std = g["profit_pct"].std()
        pair_metrics.append({
            "leader": leader,
            "follower": follower,
            "correlation": corr,
            "n_trades": n,
            "wins": wins,
            "losses": n - wins,
            "win_rate": wins / n,
            "mean_profit_pct": g["profit_pct"].mean(),
            "median_profit_pct": g["profit_pct"].median(),
            "std_profit_pct": std,
            "gross_profit_pct": gross_profit,
            "gross_loss_pct": gross_loss,
            "profit_factor": gross_profit / gross_loss if gross_loss > 0 else np.inf,
            "expected_value_pct": g["profit_pct"].mean(),
            "information_coefficient": ic,
            "max_drawdown_pct": _max_drawdown(g["profit_pct"]),
            "sharpe": (g["profit_pct"].mean() / std * np.sqrt(n)) if std > 0 else 0,
        })

    metrics = pd.DataFrame(pair_metrics)
    if metrics.empty:
        return metrics
    metrics["composite_score"] = metrics["information_coefficient"] * metrics["win_rate"]
    return metrics.sort_values("composite_score", ascending=False)


# =============================================================================
# Robust ranking (Fisher z-transform)
# =============================================================================

def add_robust_ranking(metrics: pd.DataFrame, min_trades_robust: int = 30) -> pd.DataFrame:
    """
    Add a robust ranking using the Fisher z-transform of the IC.

    Adds ``ic_lo_95``, ``ic_hi_95`` (95% CI on the information coefficient)
    and ``robust_score`` = IC * win_rate * sqrt(n_trades).

    Returns the input DataFrame with the new columns merged in.
    """
    robust = metrics[metrics["n_trades"] >= min_trades_robust].copy()
    if robust.empty:
        return metrics

    r = robust["information_coefficient"].clip(-0.999999, 0.999999)
    z = 0.5 * np.log((1 + r) / (1 - r))
    se = 1 / np.sqrt(robust["n_trades"] - 3)
    z_crit = norm.ppf(0.975)

    robust["ic_lo_95"] = (np.exp(2 * (z - z_crit * se)) - 1) / (np.exp(2 * (z - z_crit * se)) + 1)
    robust["ic_hi_95"] = (np.exp(2 * (z + z_crit * se)) - 1) / (np.exp(2 * (z + z_crit * se)) + 1)
    robust["robust_score"] = (
        robust["information_coefficient"] * robust["win_rate"] * np.sqrt(robust["n_trades"])
    )

    merge_cols = ["leader", "follower", "correlation", "ic_lo_95", "ic_hi_95", "robust_score"]
    return metrics.merge(robust[merge_cols], on=["leader", "follower", "correlation"], how="left")


def top_pairs(metrics: pd.DataFrame, metric: str = "robust_score", n: int = 20) -> pd.DataFrame:
    """Return the top ``n`` pairs ranked by ``metric``."""
    return metrics.nlargest(n, metric)


# =============================================================================
# Reporting
# =============================================================================

REPORT_COLUMNS = [
    "leader", "follower", "n_trades", "information_coefficient", "ic_lo_95",
    "win_rate", "expected_value_pct", "profit_factor", "sharpe",
    "max_drawdown_pct", "robust_score",
]


def export_report(
    metrics: pd.DataFrame,
    n: int = 20,
    output_dir: str | Path = ".",
) -> tuple[Path, Path]:
    """
    Export top ``n`` pairs to CSV and a standalone HTML report.

    Returns ``(csv_path, html_path)``.
    """
    top = metrics.nlargest(n, "robust_score")
    cols = [c for c in REPORT_COLUMNS if c in top.columns]

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    csv_path = output_dir / f"top_{n}_pairs_report.csv"
    top.to_csv(csv_path, index=False)

    html_path = output_dir / f"top_{n}_pairs_report.html"
    html = f"""
    <html><head><title>Top {n} Pairs Report</title>
    <style>
    body {{ font-family: Arial, sans-serif; margin: 40px; }}
    table {{ border-collapse: collapse; width: 100%; }}
    th, td {{ border: 1px solid #ddd; padding: 8px; text-align: right; }}
    th {{ background: #f0f0f0; }}
    td:first-child, td:nth-child(2) {{ text-align: left; }}
    </style></head><body>
    <h1>Top {n} Pairs by Robust Score</h1>
    <p>Generated: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M')}</p>
    {top[cols].to_html(float_format=lambda x: f'{x:.4f}', index=False)}
    </body></html>
    """
    html_path.write_text(html)
    return csv_path, html_path


# =============================================================================
# Pipeline (notebook 04-02)
# =============================================================================

def load_backtest_trades(cfg: Config, backtest_path: str | None = None) -> pd.DataFrame:
    """Load all backtest trades from S3 (DuckDB httpfs) or a local path."""
    backtest_path = backtest_path or (
        f"s3://{cfg.s3_bucket}/{cfg.backtest_prefix}/*/*"
    )
    con = (
        duckdb_s3_connect(cfg)
        if backtest_path.startswith("s3://")
        else duckdb.connect()
    )
    df = con.execute(
        f"SELECT * FROM read_parquet('{backtest_path}', union_by_name = true)"
    ).df()
    con.close()
    return df








In [ ]:
# ============================================================================
# Load backtest results from S3 (DuckDB httpfs)
# ============================================================================

con = duckdb_s3_connect(cfg)

backtest_path = resolve(cfg.backtest_prefix, "*/*", use_glob=True)
print("backtest_path:", backtest_path)

df = con.execute(f"""
    SELECT *
    FROM read_parquet('{backtest_path}', union_by_name = true)
""").df()

print(f"Loaded {df.shape[0]:,} backtest trades")
print(f"Unique pairs: {df.groupby(['leader','follower']).ngroups:,}")
print(f"Date range: {df['signal_date'].min()} to {df['signal_date'].max()}")
df.head(3)

In [ ]:
# ============================================================================
# Per-pair signal quality metrics
# ============================================================================

MIN_TRADES = 10
metrics = compute_pair_metrics(df, min_trades=MIN_TRADES)

print(f"Pairs with >= {MIN_TRADES} trades: {len(metrics)}")
print(f"Median IC across all pairs: {metrics['information_coefficient'].median():.4f}")
print(f"Pairs with positive IC: {(metrics['information_coefficient'] > 0).sum()} / {len(metrics)}")
print(f"Pairs with win_rate > 50%: {(metrics['win_rate'] > 0.5).sum()} / {len(metrics)}")
display(metrics)

In [ ]:
# ============================================================================
# Rank pairs by different criteria
# ============================================================================

def show_ranked(df2, metric, n=15, title=""):
    top = df2.nlargest(n, metric)
    cols = ['leader', 'follower', 'correlation', 'n_trades',
            'information_coefficient', 'win_rate', 'mean_profit_pct',
            'expected_value_pct', 'profit_factor', 'composite_score']
    cols = [c for c in cols if c in top.columns]
    print(f"\n{'=' * 70}")
    print(f"{title}  (top {len(top)} by {metric})")
    print(f"{'=' * 70}")
    display(top[cols])


show_ranked(metrics, 'composite_score', 15,
            'Top pairs by Composite Score (IC x Win Rate)')
show_ranked(metrics, 'expected_value_pct', 15,
            'Top pairs by Expected Value %')
show_ranked(metrics, 'information_coefficient', 15,
            'Top pairs by Information Coefficient')

In [ ]:
# ============================================================================
# Distribution analysis
# ============================================================================

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# IC distribution
axes[0].hist(metrics['information_coefficient'], bins=30,
             color='steelblue', edgecolor='white')
axes[0].axvline(metrics['information_coefficient'].median(),
                color='red', linestyle='--',
                label=f"Median: {metrics['information_coefficient'].median():.3f}")
axes[0].axvline(0, color='black', linewidth=0.8)
axes[0].set_title('Information Coefficient Distribution')
axes[0].set_xlabel('IC (corr of leader_gain vs profit_pct)')
axes[0].set_ylabel('Number of Pairs')
axes[0].legend()

# Win rate distribution
axes[1].hist(metrics['win_rate'], bins=30,
             color='coral', edgecolor='white')
axes[1].axvline(metrics['win_rate'].median(),
                color='red', linestyle='--',
                label=f"Median: {metrics['win_rate'].median():.1%}")
axes[1].axvline(0.5, color='black', linewidth=0.8)
axes[1].set_title('Win Rate Distribution')
axes[1].set_xlabel('Win Rate')
axes[1].set_ylabel('Number of Pairs')
axes[1].legend()

# IC vs Win Rate scatter (bubble size = n_trades)
sc = axes[2].scatter(metrics['information_coefficient'],
                     metrics['win_rate'],
                     s=metrics['n_trades'] * 2,
                     alpha=0.5,
                     c=metrics['expected_value_pct'],
                     cmap='RdYlGn',
                     edgecolors='gray', linewidth=0.5)
axes[2].axvline(0, color='black', linewidth=0.8)
axes[2].axhline(0.5, color='black', linewidth=0.8, linestyle='--')
axes[2].set_title('IC vs Win Rate (size=n_trades, color=expected_value)')
axes[2].set_xlabel('Information Coefficient')
axes[2].set_ylabel('Win Rate')
plt.colorbar(sc, ax=axes[2], label='Expected Value %')

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================================
# Robust ranking with Fisher z-confidence interval
# ============================================================================

from scipy.stats import norm

metrics = add_robust_ranking(metrics, min_trades_robust=30)

top_robust = metrics.nlargest(15, 'ic_lo_95')
print(f"\nTOP PAIRS BY IC LOWER BOUND (95% CI, n >= 30)")
display(top_robust[['leader', 'follower', 'correlation', 'n_trades',
                    'information_coefficient', 'ic_lo_95', 'ic_hi_95',
                    'win_rate', 'mean_profit_pct', 'robust_score']])

In [ ]:
# ============================================================================
# Save metrics to S3
# ============================================================================

from datetime import datetime

metrics_out = metrics.copy()
for col in metrics_out.columns:
    if pd.api.types.is_datetime64_any_dtype(metrics_out[col]):
        metrics_out[col] = metrics_out[col].astype(str)

run_timestamp = datetime.now().strftime("%Y%m%d_%H%M")
metrics_out["run_timestamp"] = run_timestamp

file_key = f"{cfg.backtest_metrics_prefix}/{run_timestamp}/data.parquet"
upload_parquet(metrics_out, s3, cfg.s3_bucket, file_key)
print(f"Saved {len(metrics_out):,} pairs -> s3://{cfg.s3_bucket}/{file_key}")
display(metrics_out.head(5)[['leader', 'follower', 'information_coefficient',
                             'win_rate', 'mean_profit_pct', 'n_trades']])